# First churn model: logistic regression

Notebook 03 turned each customer's earlier history into one row. Here we use those rows to estimate the chance of churn, then check the estimates on customers the model did not fit on.

**Scope:** a retrospective experiment with the 20,000 customers sampled from `train_v2.csv`. Notebook 04 supports treating this release as the March expiry cohort. We use features dated **1 March 2017** and supplied outcomes that were recorded later. This experiment is run after those outcomes are known; it does not simulate having a fitted model on 1 March.

## 1. Load the saved feature and label tables

Run locally in the project kernel after installing `python -m pip install -e ".[notebooks,model]"`. If the exports are missing, run notebook 03 first. This notebook reads the small saved tables and does not scan any raw archives.

Check that both files contain the same customers, that the feature date and source versions match, and that labels and customer IDs stay out of the predictor columns. The feature prototype still has open eligibility and source availability questions. Its `ready_for_training` flag is left unchanged; this notebook uses the exports for this specific retrospective experiment.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
import sklearn
from IPython.display import display

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "pyproject.toml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from churn_retention.baseline import (
    SPLIT_SEED,
    logistic_baseline,
    prepare_sample,
    probability_metrics,
    split_customers,
)

OUTPUT = ROOT / "data" / "processed"
feature_metadata = json.loads((OUTPUT / "feature_metadata.json").read_text())
features = pd.read_csv(OUTPUT / "customer_features.csv", dtype={"customer_id": "string"})
labels = pd.read_csv(OUTPUT / "supplied_labels_sample.csv", dtype={"customer_id": "string"})
X, y = prepare_sample(features, labels, feature_metadata)
print(f"{len(X):,} customers, {X.shape[1]} predictors, date {feature_metadata['prediction_date']}")

## 2. Split the customers

Use **60% for training**, **20% for validation**, and **20% for the final test**. Every customer appears in exactly one group. Stratifying keeps roughly the same proportion of churners in each group; it does not rebalance the data to 50/50.

Training teaches the model. Validation is for checking model choices. The final test is held apart from fitting and model choices. This is one month of labels, so the result measures performance on other customers from the same month. The earlier EDA informed the feature choices using this cohort, so this is still an exploratory benchmark.

In [ ]:
splits = split_customers(y)
assignments = pd.concat([
    pd.DataFrame({"customer_id": ids, "split": name}) for name, ids in splits.items()
], ignore_index=True)
assert assignments["customer_id"].is_unique
assert set(assignments["customer_id"]) == set(X.index)
split_summary = pd.DataFrame({
    name: {"customers": len(ids), "churners": int(y.loc[ids].sum()),
           "churn_rate": float(y.loc[ids].mean())}
    for name, ids in splits.items()
}).T.astype({"customers": int, "churners": int})
display(split_summary)

## 3. Fit the reference and the model on training customers

The reference predicts the **training churn rate for everyone**. Logistic regression learns how the 15 features change that probability. Its settings are fixed in advance (`C=1`); we do not tune them in this first run.

Missing numbers are filled with training medians, with extra indicators for columns that had missing values in training. Numerical features are scaled, and categories such as auto-renew status get indicator columns. These steps learn their settings from the training group along with the model. Validation and test use the same fitted settings.

In [ ]:
train_ids = splits["train"]
reference_probability = float(y.loc[train_ids].mean())
model = logistic_baseline()
model.fit(X.loc[train_ids], y.loc[train_ids])
iterations = int(model[-1].n_iter_.max())
if iterations >= model[-1].max_iter:
    raise RuntimeError("Logistic regression reached its iteration limit.")
print(f"Reference probability: {reference_probability:.2%}")
print(f"Logistic regression converged in {iterations} iterations.")

## 4. Check validation results

**Log loss** checks the probabilities and penalises confident wrong answers. Lower is better. **Average precision** and **ROC AUC** check ranking, with higher values better. The constant reference's average precision is the group's churn rate and its ROC AUC is 0.5.

In [ ]:
def evaluate(ids):
    outcomes = y.loc[ids]
    probabilities = model.predict_proba(X.loc[ids])[:, 1]
    return {
        "constant_reference": probability_metrics(outcomes, [reference_probability] * len(ids)),
        "logistic_regression": probability_metrics(outcomes, probabilities),
    }


validation_scores = evaluate(splits["validation"])
display(pd.DataFrame(validation_scores).T.round(4))

## 5. Score the fixed baseline on the final test

The fitted model is fixed before this step. Keep these results as the baseline and use validation for future tuning. A comparison on a later month requires a separate labelled cohort whose outcomes are observed, with the earlier training labels already available by the simulated fit date.

In [ ]:
test_scores = evaluate(splits["test"])
display(pd.DataFrame(test_scores).T.round(4))

## 6. Save the split and aggregate results

`baseline_split.csv` records each customer's assignment, and `baseline_metrics.json` records the aggregate results and settings. Both stay under ignored `data/processed/`. Keep the split fixed for later comparisons. Notebook outputs should be cleared before committing.

In [ ]:
results = {
    "evaluation": "retrospective_same_month_customer_holdout",
    "prediction_date": feature_metadata["prediction_date"],
    "label_source": "train_v2.csv",
    "label_timeline": "notebooks/04_label_timeline.ipynb",
    "sample_ids_sha256": feature_metadata["sample_ids_sha256"],
    "split_seed": SPLIT_SEED,
    "feature_columns": X.columns.tolist(),
    "split_summary": split_summary.to_dict(orient="index"),
    "reference_probability": reference_probability,
    "logistic_c": model[-1].C,
    "logistic_iterations": iterations,
    "validation": validation_scores,
    "test": test_scores,
    "pandas_version": pd.__version__,
    "scikit_learn_version": sklearn.__version__,
}
assignments.sort_values("customer_id").to_csv(OUTPUT / "baseline_split.csv", index=False)
(OUTPUT / "baseline_metrics.json").write_text(json.dumps(results, indent=2, allow_nan=False))
print("Saved baseline_split.csv and baseline_metrics.json under data/processed/.")